In [1]:
# from pyspark.sql import SparkSession
# import os

# spark = (
#     SparkSession.builder
#         .appName('Interactive Spark App')
#         .config("spark.hadoop.fs.s3a.endpoint", os.getenv('DATA_LAKE_ENDPOINT'))
#         .config("spark.hadoop.fs.s3a.access.key", os.getenv('DATA_LAKE_ACCESS_KEY_ID'))
#         .config("spark.hadoop.fs.s3a.secret.key", os.getenv('DATA_LAKE_SECRET_ACCESS_KEY'))
#         .getOrCreate()
# )

# spark



from pyspark.sql import SparkSession
import os

# Env vars you already use
ENDPOINT = os.getenv("DATA_LAKE_ENDPOINT")              # e.g. "http://minio:9000"
ACCESS_KEY = os.getenv("DATA_LAKE_ACCESS_KEY_ID")
SECRET_KEY = os.getenv("DATA_LAKE_SECRET_ACCESS_KEY")

# If your endpoint starts with http://, disable SSL in S3A
SSL_ENABLED = "true"
if ENDPOINT and ENDPOINT.startswith("http://"):
    SSL_ENABLED = "false"

spark = (
    SparkSession.builder
    .appName("Interactive Spark App (Delta + MinIO)")
    # ---- Delta Lake ----
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # ---- S3A / MinIO ----
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")  # REQUIRED for MinIO
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", SSL_ENABLED)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    # Optional: helps with some S3-compatible quirks
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/07 05:40:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!env

MONGO_HOST=mongodb
LANGUAGE=en_US:en
MONGO_JAVA_DRIVER_VERSION=5.6.2
MPLBACKEND=module://matplotlib_inline.backend_inline
HOSTNAME=72ae659331b8
SHLVL=0
HOME=/root
KAFKA_CLIENT_VERSION=3.9.1
KAFKA_BOOTSTRAP=broker:29092
MONGO_PORT=27017
SPARK_BUFFER_SIZE=65536
GPG_KEY=F28C9C925C188C35E345614DEDA00CE834F0FC5C
PAGER=cat
SPARK_TGZ_ASC_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz.asc?action=download
MONGO_SPARK_VERSION=10.5.0
KAFKA_VERSION=4.0.1
SPARK_TGZ_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz?action=download
JAVA_VERSION=jdk-17.0.16+8
AWS_SDK_V1_VERSION=1.12.793
AWS_SDK_V2_VERSION=2.40.13
FORCE_COLOR=1
HADOOP_VERSION=3.4.0
MONGO_INITDB_ROOT_PASSWORD=password
SPARK_VERSION=4.0.1
TERM=xterm-color
MONGO_APP_DB=yelp
PATH=/opt/java/openjdk/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
MONGO_AUTH_DB=admin
LANG=en_US.UTF-8
CLICOLOR_FORCE=1
DELTA_VERSION=4.0.0
SCALA_VERSION=2.13
GIT_PAGER

In [4]:
spark.conf.get('spark.hadoop.fs.s3a.access.key')

'admin'

In [5]:
# Sample data (list of tuples: name, age)
data = [("Alice", 25), ("Bob", 17), ("Charlie", 32), ("Diana", 12)]

# Define schema for the DataFrame
columns = ["Name", "Age"]

# Create a Spark DataFrame
df = spark.createDataFrame(data, columns)

# Display the original DataFrame
print("Original DataFrame:")
df.show()

Original DataFrame:


+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 32|
|  Diana| 12|
+-------+---+



In [6]:
delta_path = "s3a://bronze/demo/people_delta"

(
    df.write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

# df.coalesce(1).write.format("delta").mode("overwrite").save(delta_path)

print("Wrote Delta to:", delta_path)

26/01/07 05:41:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/01/07 05:42:12 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to demo/people_delta/part-00000-d5c77b6f-3828-45d2-8547-e557d39ad943-c000.snappy.parquet. This is Unsupported
                                                                                

Wrote Delta to: s3a://bronze/demo/people_delta


In [7]:
df = spark.read.format("delta").load(delta_path)
df.printSchema()
print("Row count:", df.count())
df.show(20, truncate=False)

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)



26/01/07 05:42:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Row count: 4


+-------+---+
|Name   |Age|
+-------+---+
|Charlie|32 |
|Alice  |25 |
|Diana  |12 |
|Bob    |17 |
+-------+---+



In [8]:
df.rdd.getNumPartitions()

4

In [9]:
from pyspark.sql.functions import spark_partition_id

(
    df
    .withColumn("spark_partition_id", spark_partition_id())
    .orderBy("spark_partition_id")
    .show(truncate=False)
)


+-------+---+------------------+
|Name   |Age|spark_partition_id|
+-------+---+------------------+
|Charlie|32 |0                 |
|Alice  |25 |1                 |
|Diana  |12 |2                 |
|Bob    |17 |3                 |
+-------+---+------------------+



In [2]:
print("Spark:", spark.version)
print("spark.sql.extensions:", spark.conf.get("spark.sql.extensions", ""))
print("spark.sql.catalog.spark_catalog:", spark.conf.get("spark.sql.catalog.spark_catalog", ""))
print("JARS:", spark.sparkContext.getConf().get("spark.jars", ""))


Spark: 4.0.1
spark.sql.extensions: io.delta.sql.DeltaSparkSessionExtension
spark.sql.catalog.spark_catalog: org.apache.spark.sql.delta.catalog.DeltaCatalog
JARS: 
